# 06 - Praproses, Pelatihan Model, Tuning, dan Prediksi

Untuk setiap ambang (`0.5` utama; `0.25` dan `1.0` sensitivitas), periode uji, model (Regresi Logistik Multinomial & XGBoost), dan konfigurasi fitur (`K1`, `K2`, `K3`):

1. **Tuning (*validasi internal*):** setiap kandidat *hyperparameter* dilatih dan diuji pada 3 blok validasi internal, lalu dipilih yang rata-rata Macro-F1-nya tertinggi.
2. **Pelatihan akhir:** *hyperparameter* terbaik dilatih ulang pada seluruh data latih, lalu dipakai memprediksi periode uji.
3. **Parameter tetap K1:** *hyperparameter* terbaik K1 dipakai ulang untuk K2 dan K3 sebagai pengujian pembanding.
4. **Pembanding kelas mayoritas:** memprediksi kelas terbanyak pada data latih.

Hasil prediksi disimpan ke `06_prediksi.parquet` untuk dievaluasi pada notebook 07.

In [1]:
import itertools
import time
import warnings

import numpy as np
import pandas as pd
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

DATA_DIR = "../data/interim/phase4-p0"
SEED = 42
AMBANG = [0.5, 0.25, 1.0]
KELAS = ["SELL", "HOLD", "BUY"]            # 0 = SELL, 1 = HOLD, 2 = BUY
URUT_SERI_MAYORITAS = ["BUY", "HOLD", "SELL"]

# Ubah ke list tertentu kalau mau coba cepat, misal ["2023-H1"]. None = jalankan semua.
PERIODE_DIJALANKAN = None
AMBANG_DIJALANKAN = None

K1 = [
    "return_lag_0", "return_lag_1", "return_lag_2",
    "return_lag_3", "return_lag_4", "return_lag_5",
    "log_close_open", "log_high_low", "sigma_24",
]
TEKNIKAL = ["rsi_14", "macd_line_norm", "macd_signal_norm", "macd_hist_norm"]
K2 = K1 + TEKNIKAL

In [2]:
df = pd.read_parquet(f"{DATA_DIR}/05_dataset.parquet")
blok = pd.read_csv(f"{DATA_DIR}/05_blok.csv", parse_dates=["awal", "akhir", "keputusan_pertama"])

with open(f"{DATA_DIR}/04_status_k3.txt", encoding="utf-8") as f:
    STATUS_K3 = f.read().strip()

K3 = K2 + [c for c in df.columns if c.startswith("macro_")]
KONFIGURASI = {"K1": K1, "K2": K2}
if STATUS_K3 == "LAYAK":
    KONFIGURASI["K3"] = K3

print("Jumlah fitur per konfigurasi:", {k: len(v) for k, v in KONFIGURASI.items()})
print("Status K3:", STATUS_K3)

Konfigurasi: {'K1': 9, 'K2': 13, 'K3': 59} · status K3: LAYAK


## 1. Daftar Kandidat Hyperparameter

In [3]:
KANDIDAT = {
    "regresi_logistik": [{"C": c} for c in [0.01, 0.1, 1.0, 10.0]],
    "xgboost": [
        {"max_depth": d, "n_estimators": n, "learning_rate": lr}
        for d, n, lr in itertools.product([3, 5], [100, 300], [0.05, 0.1])
    ],
}

def urutan_seri(model, param):
    # Jika rata-rata Macro-F1 sama persis, pilih model yang lebih sederhana
    if model == "regresi_logistik":
        return (param["C"],)
    return (param["max_depth"], param["n_estimators"], param["learning_rate"])

daftar_tabel = []
for nama_model, daftar_param in KANDIDAT.items():
    for p in daftar_param:
        daftar_tabel.append({"model": nama_model, **p})
pd.DataFrame(daftar_tabel)

,model,C,max_depth,n_estimators,learning_rate
0,regresi_logistik,0.01,NaN,NaN,NaN
1,regresi_logistik,0.10,NaN,NaN,NaN
2,regresi_logistik,1.00,NaN,NaN,NaN
3,regresi_logistik,10.00,NaN,NaN,NaN
4,xgboost,NaN,3.0,100.0,0.05
5,xgboost,NaN,3.0,100.0,0.10
6,xgboost,NaN,3.0,300.0,0.05
7,xgboost,NaN,3.0,300.0,0.10
8,xgboost,NaN,5.0,100.0,0.05
9,xgboost,NaN,5.0,100.0,0.10


## 2. Praproses Fitur (Hanya Belajar dari Data Latih)

1. Buang kolom yang seluruhnya kosong di data latih.
2. Isi nilai kosong memakai **median data latih**, lalu tambahkan penanda biner nilai kosong.
3. Standardisasi fitur (`StandardScaler`) dihitung dari data latih (dipakai di Regresi Logistik, dan di K3 untuk menyamakan satuan fitur makro).

In [4]:
class Praproses:
    def __init__(self, pakai_scaler):
        self.pakai_scaler = pakai_scaler

    def fit(self, data_latih):
        self.dipakai = [c for c in data_latih.columns if data_latih[c].notna().any()]
        self.dibuang = [c for c in data_latih.columns if c not in self.dipakai]
        self.median = data_latih[self.dipakai].median()
        self.kolom_kosong = [c for c in self.dipakai if data_latih[c].isna().any()]

        if self.pakai_scaler:
            self.scaler = StandardScaler().fit(self._bentuk_matriks(data_latih))
        else:
            self.scaler = None
        return self

    def _bentuk_matriks(self, data):
        nilai = data[self.dipakai].fillna(self.median).to_numpy(float)
        indikator = data[self.kolom_kosong].isna().to_numpy(float)
        return np.column_stack([nilai, indikator])

    def transform(self, data):
        x = self._bentuk_matriks(data)
        if self.scaler is not None:
            x = self.scaler.transform(x)
        return x

## 3. Fungsi Pelatihan Model dan Prediksi

Bobot kelas seimbang dihitung dari data latih: $w_c = n / (3 n_c)$.

In [5]:
class GagalEvaluasi(Exception):
    pass

def latih_prediksi(model, param, latih, uji, kolom, kolom_label):
    peta_kelas = {k: i for i, k in enumerate(KELAS)}
    y = latih[kolom_label].map(peta_kelas).to_numpy()
    n_kelas = np.bincount(y, minlength=3)
    if (n_kelas == 0).any():
        raise GagalEvaluasi("data latih tidak memuat tiga kelas")

    pakai_scaler = (model == "regresi_logistik") or (kolom is K3)
    prep = Praproses(pakai_scaler=pakai_scaler).fit(latih[kolom])
    x_latih = prep.transform(latih[kolom])
    x_uji = prep.transform(uji[kolom])
    bobot = len(y) / (3 * n_kelas)

    if model == "regresi_logistik":
        clf = LogisticRegression(
            C=param["C"],
            solver="lbfgs",
            max_iter=2000,
            class_weight={i: bobot[i] for i in range(3)},
        )
        with warnings.catch_warnings():
            warnings.simplefilter("error", ConvergenceWarning)
            try:
                clf.fit(x_latih, y)
            except ConvergenceWarning as e:
                raise GagalEvaluasi(f"tidak konvergen: {e}") from e
    else:
        clf = XGBClassifier(
            objective="multi:softprob",
            num_class=3,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_lambda=1.0,
            random_state=SEED,
            n_jobs=1,
            tree_method="hist",
            eval_metric="mlogloss",
            **param,
        )
        clf.fit(x_latih, y, sample_weight=bobot[y])

    pred_angka = clf.predict(x_uji).astype(int)
    pred_label = np.array(KELAS)[pred_angka]
    return pred_label, prep.dibuang

def hitung_macro_f1(benar, prediksi):
    return f1_score(benar, prediksi, labels=KELAS, average="macro", zero_division=0)

## 4. Fungsi Potong Data dan Tuning Validasi Internal

In [6]:
def potong_data(periode, nama_blok):
    r = blok[(blok["periode"] == periode) & (blok["blok"] == nama_blok)].iloc[0]
    latih = df[df["target_end_time"] < r["keputusan_pertama"]]
    uji = df[
        (df["decision_time"] >= r["awal"])
        & (df["decision_time"] < r["akhir"])
        & (df["target_end_time"] < r["akhir"])
    ]
    return latih, uji

def pilih_hyperparameter(model, kolom, kolom_label, periode, catatan):
    hasil_valid = []

    for param in KANDIDAT[model]:
        skor_blok = []
        alasan_gagal = None

        for nama_blok in ("validasi-1", "validasi-2", "validasi-3"):
            latih, valid = potong_data(periode, nama_blok)
            try:
                pred, _ = latih_prediksi(model, param, latih, valid, kolom, kolom_label)
                skor_blok.append(hitung_macro_f1(valid[kolom_label], pred))
            except GagalEvaluasi as e:
                alasan_gagal = f"{nama_blok}: {e}"
                break

        rata = float(np.mean(skor_blok)) if alasan_gagal is None else None
        catatan.append({**param, "model": model, "skor_validasi": skor_blok, "rata_validasi": rata, "gagal": alasan_gagal})

        if rata is not None:
            hasil_valid.append((rata, param))

    if not hasil_valid:
        return None

    terbaik = min(hasil_valid, key=lambda h: (-h[0], *urutan_seri(model, h[1])))
    return terbaik[1]

## 5. Jalankan Pelatihan Seluruh Periode Uji

Untuk seluruh 3 ambang × 7 periode uji, proses ini memakan waktu sekitar 15–20 menit (karena `n_jobs=1` agar hasil XGBoost tetap konsisten).

In [7]:
def cari_kelas_mayoritas(latih, kolom_label):
    hitung = latih[kolom_label].value_counts()
    return min(URUT_SERI_MAYORITAS, key=lambda k: (-hitung.get(k, 0), URUT_SERI_MAYORITAS.index(k)))

daftar_prediksi = []
daftar_kandidat = []
daftar_kegagalan = []
mulai = time.time()

for ambang in AMBANG:
    if AMBANG_DIJALANKAN and ambang not in AMBANG_DIJALANKAN:
        continue
    kolom_label = "label_" + str(ambang).replace(".", "p")

    for periode in blok["periode"].unique():
        if PERIODE_DIJALANKAN and periode not in PERIODE_DIJALANKAN:
            continue

        latih, uji = potong_data(periode, "uji")
        dasar = {
            "ambang": ambang,
            "periode": periode,
            "time": uji["time"].to_numpy(),
            "benar": uji[kolom_label].to_numpy(),
        }

        # Pembanding kelas mayoritas
        kelas_mayor = cari_kelas_mayoritas(latih, kolom_label)
        daftar_prediksi.append(pd.DataFrame({
            **dasar, "model": "mayoritas", "konfigurasi": "-",
            "analisis": "diagnostik", "prediksi": kelas_mayor,
        }))

        # Pelatihan Regresi Logistik & XGBoost
        for model in KANDIDAT:
            param_k1 = None

            for konfig, kolom in KONFIGURASI.items():
                catatan = []
                param = pilih_hyperparameter(model, kolom, kolom_label, periode, catatan)
                for c in catatan:
                    daftar_kandidat.append({**c, "ambang": ambang, "periode": periode, "konfigurasi": konfig})

                if param is None:
                    daftar_kegagalan.append({
                        "ambang": ambang, "periode": periode, "model": model,
                        "konfigurasi": konfig, "alasan": "semua kandidat gagal pada validasi internal",
                    })
                    continue

                try:
                    pred, dibuang = latih_prediksi(model, param, latih, uji, kolom, kolom_label)
                except GagalEvaluasi as e:
                    daftar_kegagalan.append({
                        "ambang": ambang, "periode": periode, "model": model,
                        "konfigurasi": konfig, "alasan": str(e),
                    })
                    continue

                info = {"model": model, "konfigurasi": konfig, "parameter": str(param), "kolom_dibuang": len(dibuang)}
                daftar_prediksi.append(pd.DataFrame({**dasar, **info, "analisis": "disetel", "prediksi": pred}))

                # Pengujian pembanding: parameter K1 dipakai ulang di K2 dan K3
                if konfig == "K1":
                    param_k1 = param
                elif param_k1 is not None:
                    pred_tetap, _ = latih_prediksi(model, param_k1, latih, uji, kolom, kolom_label)
                    daftar_prediksi.append(pd.DataFrame({
                        **dasar, **info, "parameter": str(param_k1),
                        "analisis": "parameter_tetap_k1", "prediksi": pred_tetap,
                    }))

                print(f"{ambang:<5} {periode:13} {model:17} {konfig}  terpilih {param}  ({time.time() - mulai:.0f} dtk)")

0.5   2023-H1       regresi_logistik  K1  terpilih {'C': 0.01}  (0 dtk)


0.5   2023-H1       regresi_logistik  K2  terpilih {'C': 0.1}  (1 dtk)


0.5   2023-H1       regresi_logistik  K3  terpilih {'C': 10.0}  (1 dtk)


0.5   2023-H1       xgboost           K1  terpilih {'max_depth': 3, 'n_estimators': 100, 'learning_rate': 0.05}  (7 dtk)


0.5   2023-H1       xgboost           K2  terpilih {'max_depth': 3, 'n_estimators': 100, 'learning_rate': 0.05}  (15 dtk)


0.5   2023-H1       xgboost           K3  terpilih {'max_depth': 3, 'n_estimators': 100, 'learning_rate': 0.05}  (30 dtk)


... (116 outputs left out)

1.0   2026-Jan-Jul  regresi_logistik  K3  terpilih {'C': 0.01}  (880 dtk)


1.0   2026-Jan-Jul  xgboost           K1  terpilih {'max_depth': 5, 'n_estimators': 300, 'learning_rate': 0.05}  (891 dtk)


1.0   2026-Jan-Jul  xgboost           K2  terpilih {'max_depth': 5, 'n_estimators': 300, 'learning_rate': 0.1}  (906 dtk)


1.0   2026-Jan-Jul  xgboost           K3  terpilih {'max_depth': 5, 'n_estimators': 300, 'learning_rate': 0.1}  (938 dtk)


In [8]:
prediksi = pd.concat(daftar_prediksi, ignore_index=True)
kandidat = pd.DataFrame(daftar_kandidat)
kandidat["skor_validasi"] = kandidat["skor_validasi"].map(lambda s: ";".join(f"{v:.12f}" for v in s))

prediksi.to_parquet(f"{DATA_DIR}/06_prediksi.parquet", index=False)
kandidat.to_parquet(f"{DATA_DIR}/06_kandidat.parquet", index=False)
pd.DataFrame(daftar_kegagalan, columns=["ambang", "periode", "model", "konfigurasi", "alasan"]).to_csv(
    f"{DATA_DIR}/06_kegagalan.csv", index=False
)

print("Baris prediksi:", len(prediksi), "| kandidat dicatat:", len(kandidat), "| kegagalan:", len(daftar_kegagalan))
print("Kandidat gagal pada validasi internal:", int(kandidat["gagal"].notna().sum()))

Baris prediksi: 656469 · kandidat dicatat: 756 · kegagalan: 0
Kandidat gagal pada validasi internal: 0
